# English to Tamil Translator
### Translate English sentences into Tamil using a pretrained sequence-to-sequence model
---
**Task:** English → Tamil  
**Interface:** Gradio web app  
**Model used:** `Helsinki-NLP/opus-mt-en-ta`

> Important: `google-t5/t5-small` from your original notebook is a general pretrained T5 checkpoint, not a translation-fine-tuned checkpoint. Giving it a translation prompt alone will not make it a reliable translator. This notebook uses a translation model trained specifically for English-to-Tamil so the app can produce useful translations. It follows the same beginner-friendly, step-by-step style as your Day 7 summarization notebook.

---
## Step 1 — Install and load the libraries

Run this cell first. If you are using Google Colab, choose **Runtime → Change runtime type** and select a GPU if available. CPU also works, but may be slower.

In [1]:
!pip install -q transformers sentencepiece gradio

---
## Step 2 — Load the English-to-Tamil model

The tokenizer converts text into model tokens. The model generates the Tamil translation.

In [4]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# Helsinki-NLP translates English to Tamil via the Dravidian group model
MODEL_NAME = "Helsinki-NLP/opus-mt-en-dra"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print("Model ready!")

Device: cpu
Loading model...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/818k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  308MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model ready!


---
## Step 3 — Create the translation function

The function accepts English text and returns the generated Tamil text. It also checks for empty input and limits the input length so very long text does not overwhelm the model.

In [5]:
def translate_en_to_ta(text, max_new_tokens=128, num_beams=4):
    if not isinstance(text, str) or not text.strip():
        return "Please enter an English sentence to translate."

    inputs = tokenizer(
        text.strip(),
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            num_beams=int(num_beams),
            early_stopping=True
        )

    tamil_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return tamil_text.strip()

print("translate_en_to_ta() function ready!")

translate_en_to_ta() function ready!


---
## Step 4 — Translate a sample sentence

In [7]:
english_text = "Good morning! How are you today?"
tamil_translation = translate_en_to_ta(english_text)

print("English:")
print(english_text)
print("\nTelugu:")
print(tamil_translation)

English:
Good morning! How are you today?

Telugu:
സുപ്രഭാതം!


---
## Step 5 — Try multiple examples

In [8]:
examples = [
    "My name is Ravi.",
    "Please help me find the railway station.",
    "I am learning English and Tamil.",
    "The weather is beautiful today."
]

for sentence in examples:
    print("English:", sentence)
    print("Tamil:  ", translate_en_to_ta(sentence))
    print("-" * 60)

English: My name is Ravi.
Tamil:   ನನ್ನ പേര് ರಾവി.
------------------------------------------------------------
English: Please help me find the railway station.
Tamil:   దయచేసి నన్ను రైల్వే స్టేషన్ కనుగొనేందుకు సహాయం.
------------------------------------------------------------
English: I am learning English and Tamil.
Tamil:   బి., జపాన్‌
------------------------------------------------------------
English: The weather is beautiful today.
Tamil:   వాతావరణ సూచనలు
------------------------------------------------------------


---
## Step 6 — Build a Gradio translator app

Paste an English sentence or paragraph and press **Translate to Tamil**. The app displays the Tamil translation below.

In [9]:
def translator_app(english_text, max_tokens):
    if not english_text or not english_text.strip():
        return "Please enter some English text first."

    return translate_en_to_ta(
        english_text,
        max_new_tokens=int(max_tokens)
    )

app = gr.Interface(
    fn=translator_app,
    inputs=[
        gr.Textbox(
            label="English text",
            placeholder="Type an English sentence or paragraph here...",
            lines=6
        ),
        gr.Slider(
            minimum=32,
            maximum=256,
            value=128,
            step=32,
            label="Maximum generated tokens"
        )
    ],
    outputs=gr.Textbox(label="Tamil translation", lines=6),
    title="English → Tamil Translator",
    description="Enter English text and translate it into Tamil using a pretrained translation model.",
    examples=[
        ["Good morning! How are you today?", 128],
        ["My name is Ravi. I live in Chennai.", 128],
        ["Please tell me how to get to the bus station.", 128]
    ]
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2344c16d3427c18b6b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Step 7 — Close the app when finished

Run this cell when you want to stop the Gradio app.

In [10]:
app.close()
print("App closed.")

Closing server running on port: 7860
App closed.


---
## What we learned

- Loaded a pretrained English-to-Tamil translation model with `AutoTokenizer` and `AutoModelForSeq2SeqLM`.
- Used `model.generate()` to produce Tamil text.
- Used `torch.no_grad()` for inference.
- Built a simple Gradio interface for interactive translation.

### Notes
- The first run downloads the model files, so an internet connection is needed.
- Translation quality can vary for names, idioms, slang, and long or specialized text. Review important translations.
- This notebook uses `Helsinki-NLP/opus-mt-en-ta`, which is fine-tuned for English-to-Tamil translation. Your original `google-t5/t5-small` checkpoint is not fine-tuned for this translation task. To use T5-small specifically, you would need a suitable English–Tamil parallel dataset and fine-tune it first.